In [9]:
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from Plotting_IQR import plot_distribution, get_training_data, plot_pfn_variance_surface, plot_GP_variance_surface
from pfn_evaluate import eval_pfn
import pfns4bo
from pfns4bo.scripts.acquisition_functions import TransformerBOMethod

In [ ]:
# Load data
file_path = "dim_varied_results/run_20260804_112003/metrics.pt"
metrics = torch.load(file_path)

file_path = "dim_varied_results/run_20260804_112003/experimental_results.pt"
data = torch.load(file_path)

In [ ]:
#metrics = {
#        "pred_error": torch.zeros((n_tests, n_dims, n_methods, n_repeats, n_samples, n_fns)), # GP & PFN only
#        "total_error": torch.zeros((n_tests, n_dims, n_methods, n_repeats, 1)),
#        "EI": torch.zeros((n_tests, n_dims, n_methods, n_repeats, n_samples, 1))
#    }
pred_error = metrics["pred_error"]
total_error = metrics["total_error"]
ei = metrics["EI"]
gll = metrics["GLL"]
m_gll = metrics["MGLL"]

y_true_store = data["y_true"][0]
mu_store = data["mu"][0]
var_store = data["var"][0]
x_queried = data["x_query"][0]

In [ ]:
mu_data_GP = torch.sum(mu_store[:, 0, 0, :, :], dim=-1, keepdim=True)
mu_data_PFN = torch.sum(mu_store[:, 1, 0, :, :], dim=-1, keepdim=True)
mu_data_PFN_W = torch.sum(mu_store[:, 2, 0, :, :], dim=-1, keepdim=True)
var_data_GP = torch.sum(var_store[:, 0, 0, :, :], dim=-1, keepdim=True)
var_data_PFN = torch.sum(var_store[:, 1, 0, :, :], dim=-1, keepdim=True)
var_data_PFN_W = torch.sum(var_store[:, 2, 0, :, :], dim=-1, keepdim=True)
x_query_GP = x_queried[:, 0, 0, :, :]
x_query_PFN = x_queried[:, 1, 0, :, :]
x_query_PFN_W = x_queried[:, 2, 0, :, :]
y_true_arr = y_true_store[:, 0, 0, :, :]

In [ ]:
# metrics["pred_error"][test, k, m_idx, rep, :, :]
# 1 tests, 6 dim, 2 methods, 21 reps, 1000 samples, dim = 1 -> 1 tests, 2 methods, 1000 samples, 1 
pred_error_med = torch.quantile(torch.sum(pred_error[0, :, :, :, :, :], dim=-1, keepdim=True), 0.5, dim=-3)
total_error_med = torch.quantile(total_error[0, :, :, :, :], 0.5, dim=-3)
ei_med = torch.quantile(torch.sum(ei[0, :, :, :, :, :], dim=-1, keepdim=True), 0.5, dim=-3)

pred_error_lq = torch.quantile(torch.sum(pred_error[0, :, :, :, :, :], dim=-1, keepdim=True), 0.25, dim=-3)
total_error_lq = torch.quantile(total_error[0, :, :, :, :], 0.25, dim=-3)
ei_lq = torch.quantile(torch.sum(ei[0, :, :, :, :, :], dim=-1, keepdim=True), 0.25, dim=-3)

pred_error_uq = torch.quantile(torch.sum(pred_error[0, :, :, :, :, :], dim=-1, keepdim=True), 0.75, dim=-3)
total_error_uq = torch.quantile(total_error[0, :, :, :, :], 0.75, dim=-3)
ei_uq = torch.quantile(torch.sum(ei[0, :, :, :, :, :], dim=-1, keepdim=True), 0.75, dim=-3)

In [ ]:
# metrics["pred_error"][test, k, m_idx, rep, :, :]
# 1 tests, 6 dim, 3 methods, 21 reps, 1000 samples, dim = 1 -> 9 tests, 3 methods, 1000 samples, 1 
gll_med = torch.quantile(torch.sum(gll[0, :, :, :, :, :], dim=-1, keepdim=True), 0.5, dim=-3)
mgll_med = torch.quantile(m_gll[0, :, :, :, :], 0.5, dim=-3)

gll_lq = torch.quantile(torch.sum(gll[0, :, :, :, :, :], dim=-1, keepdim=True), 0.25, dim=-3)
mgll_lq = torch.quantile(m_gll[0, :, :, :, :], 0.25, dim=-3)

gll_uq = torch.quantile(torch.sum(gll[0, :, :, :, :, :], dim=-1, keepdim=True), 0.75, dim=-3)
mgll_uq = torch.quantile(m_gll[0, :, :, :, :], 0.75, dim=-3)

In [ ]:
mgllm_GP = mgll_med[:, 0].squeeze(-1)
mglllq_GP = mgll_lq[:, 0].squeeze(-1)
mglluq_GP = mgll_uq[:, 0].squeeze(-1)

mgllm_PFN = mgll_med[:, 1].squeeze(-1)
mglllq_PFN = mgll_lq[:, 1].squeeze(-1)
mglluq_PFN = mgll_uq[:, 1].squeeze(-1)

mgllm_PFN_W = mgll_med[:, 2].squeeze(-1)
mglllq_PFN_W = mgll_lq[:, 2].squeeze(-1)
mglluq_PFN_W = mgll_uq[:, 2].squeeze(-1)

tem_GP = total_error_med[:, 0].squeeze(-1)
telq_GP = total_error_lq[:, 0].squeeze(-1)
teuq_GP = total_error_uq[:, 0].squeeze(-1)

tem_PFN = total_error_med[:, 1].squeeze(-1)
telq_PFN = total_error_lq[:, 1].squeeze(-1)
teuq_PFN = total_error_uq[:, 1].squeeze(-1)

tem_PFN_W = total_error_med[:, 2].squeeze(-1)
telq_PFN_W = total_error_lq[:, 2].squeeze(-1)
teuq_PFN_W = total_error_uq[:, 2].squeeze(-1)

dims = torch.tensor([1, 2, 4, 6, 8, 10])

fig, (ax1, ax2) = plt.subplots(
    nrows=2,
    ncols=1,
    figsize=(10, 10),
    sharex=True,
    sharey=False,
)
ax1.fill_between(
    dims,
    telq_GP,
    teuq_GP,
    color="tab:blue",
    alpha=0.5,
    edgecolor="none",
    label="IQR",
)
ax1.plot(dims, tem_GP, color="tab:blue", linewidth=2, label="Median")
ax1.grid(True, linestyle="--", alpha=0.5)
ax1.legend(loc="upper right")

ax1.fill_between(
    dims,
    telq_PFN,
    teuq_PFN,
    color="tab:orange",
    alpha=0.5,
    edgecolor="none",
    label="IQR",
)
ax1.plot(dims, tem_PFN, color="tab:orange", linewidth=2, label="Median")
ax1.grid(True, linestyle="--", alpha=0.5)
ax1.legend(loc="upper right")

ax1.fill_between(
    dims,
    telq_PFN_W,
    teuq_PFN_W,
    color="tab:green",
    alpha=0.5,
    edgecolor="none",
    label="IQR",
)
ax1.plot(dims, tem_PFN_W, color="tab:green", linewidth=2, label="Median")
ax1.set_title(f"Prediction Error")
ax1.grid(True, linestyle="--", alpha=0.5)
ax1.legend(loc="upper right")

ax2.fill_between(
    dims,
    mglllq_GP,
    mglluq_GP,
    color="tab:blue",
    alpha=0.5,
    edgecolor="none",
    label="IQR",
)
ax2.plot(dims, mgllm_GP, color="tab:blue", linewidth=2, label="Median")
ax2.grid(True, linestyle="--", alpha=0.5)
ax2.legend(loc="upper right")

ax2.fill_between(
    dims,
    mglllq_PFN,
    mglluq_PFN,
    color="tab:orange",
    alpha=0.5,
    edgecolor="none",
    label="IQR",
)
ax2.plot(dims, mgllm_PFN, color="tab:orange", linewidth=2, label="Median")
ax2.grid(True, linestyle="--", alpha=0.5)
ax2.legend(loc="upper right")

ax2.fill_between(
    dims,
    mglllq_PFN_W,
    mglluq_PFN_W,
    color="tab:green",
    alpha=0.5,
    edgecolor="none",
    label="IQR",
)
ax2.plot(dims, mgllm_PFN_W, color="tab:green", linewidth=2, label="Median")
ax2.set_title(f"Log Likelyhood")
ax2.grid(True, linestyle="--", alpha=0.5)
ax2.legend(loc="upper right")

ax2.set_xlabel("x")
fig.text(0.01, 0.5, "Log Likelyhood", va="center", rotation="vertical")

plt.tight_layout()
plt.show()